# Getting Started
* Create a new API key at https://www.console.anthropic.com
* Create a .env file
  * Add ```ANTHROPIC_API_KEY="<your_api_key>"```
* Ensure uv is installed: https://docs.astral.sh/uv/getting-started/installation/
* Run Standalone Notebook:
  * > uv run jupyter lab
* Run Notebook in VSCode:
  * Open Jupyter Notebook and select `.venv (Python 3.12.11)` as the kernel

In [28]:

from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from pydantic import BaseModel
client = Anthropic()
model = 'claude-sonnet-4-5'
max_tokens = 1000

In [29]:
def add_user_message(messages, text):
  user_message = {'role': 'user', 'content': text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {'role': 'assistant', 'content': text}
  messages.append(assistant_message)

def chat(messages, system=None, stop_sequences=None, temperature=1.0):
  params = {
    'model': model,
    'max_tokens': max_tokens,
    'messages': messages,
    'temperature': temperature
  }

  if system:
    params['system'] = system

  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  message = client.messages.create(**params)

  return message.content[0].text


## Claude API

In [30]:
# Claude doesn't maintain chat history. Each message effectively starts from scratch.
# To give Claude memory we need to build our messages and Claude's into a conversation history. We'll keep appending to this with every request/response
messages = []

# add initial user question
add_user_message(messages, 'What is quantum computing?')

# get claude's response
answer = chat(messages)

# add claude's response to history
add_assistant_message(messages, answer)

# add a follow up question
add_user_message(messages, 'Write another sentence')

# get the follow up
final_answer = chat(messages)
print(final_answer)

Quantum computers represent one of the most promising frontiers in technology, though practical, large-scale quantum computing that outperforms classical computers for real-world applications may still be years or even decades away.


In [31]:
# Adding system prompts
messages = []

add_user_message(messages, 'Write a python function that checks a string for duplicate characters')

answer = chat(messages, system='you are an expert python developer. write clean and concise code')

print(answer)

```python
def has_duplicate_chars(s: str) -> bool:
    """
    Check if a string contains duplicate characters.
    
    Args:
        s: Input string to check
        
    Returns:
        True if duplicates exist, False otherwise
    """
    return len(s) != len(set(s))


# Alternative: Get the duplicate characters
def get_duplicate_chars(s: str) -> set:
    """
    Return a set of characters that appear more than once.
    
    Args:
        s: Input string to check
        
    Returns:
        Set of duplicate characters
    """
    seen = set()
    duplicates = set()
    
    for char in s:
        if char in seen:
            duplicates.add(char)
        else:
            seen.add(char)
    
    return duplicates


# Example usage
if __name__ == "__main__":
    test_strings = ["hello", "world", "python", "abc"]
    
    for string in test_strings:
        print(f"'{string}' has duplicates: {has_duplicate_chars(string)}")
        if has_duplicate_chars(string):
            print(

In [32]:
# low temperature - more predicable
# this can be used for more factual responses like coding
answer = chat(messages, temperature=0.0)
print(answer)

print('HIGH TEMP VERSION -------------------------------------')

# high temp - more creative
# this can be used for brainstorming
answer = chat(messages, temperature=1.0)
print(answer)

# Python Function to Check for Duplicate Characters

Here are several approaches to check for duplicate characters in a string:

## 1. Basic Approach (Returns Boolean)

```python
def has_duplicates(s):
    """
    Check if a string contains duplicate characters.
    
    Args:
        s (str): The string to check
        
    Returns:
        bool: True if duplicates exist, False otherwise
    """
    return len(s) != len(set(s))

# Example usage
print(has_duplicates("hello"))      # True (l appears twice)
print(has_duplicates("world"))      # True (o appears twice)
print(has_duplicates("python"))     # False
```

## 2. Return Duplicate Characters

```python
def find_duplicates(s):
    """
    Find all duplicate characters in a string.
    
    Args:
        s (str): The string to check
        
    Returns:
        set: Set of characters that appear more than once
    """
    seen = set()
    duplicates = set()
    
    for char in s:
        if char in seen:
            duplicates.ad

In [33]:
# Stream responses instead of waiting for the full answer
messages = []
add_user_message(messages, 'Write a 1 sentence description of a fake database')

stream = client.messages.create(
  model=model,
  max_tokens=1000,
  messages=messages,
  stream=True
)

for event in stream:
  print(event)

RawMessageStartEvent(message=Message(id='msg_01Upb7yZsevjf7exRrvtjZrd', container=None, content=[], model='claude-sonnet-4-5-20250929', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' fake database is a simulated or mock data storage system that mimics the structure and behavior of a real database but', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=Text

In [34]:
# More simplified way
messages = []
add_user_message(messages, 'Write a 1 sentence description of a fake database')

with client.messages.stream(
  model=model,
  max_tokens=1000,
  messages=messages
) as stream:
  for text in stream.text_stream:
    print(text, end='')
  # Get the complete message for database storage
  final_message = stream.get_final_message()

A fake database is a simulated or mock data storage system that mimics the structure and behavior of a real database but contains fabricated, randomized, or placeholder information used for testing, development, or demonstration purposes.

In [35]:
messages = []
add_user_message(messages, 'Generate a very short event bridge rule as json')
answer = chat(messages)

print(answer)

```json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}
```


In [36]:
# The issue with the previous block is default responses typically has some explanatory text around the JSON we're interested in.
# This creates friction if we want to directly use the JSON output directly from Claude without text cleanup.
# To solve this, we can add message prefilling and stop sequences
# In this example, it makes Claude think it already started generating a JSON block
# so it will continue to do so until it reaches the stop sequence we provide
# Note: I noticed prefills dont work with sonnet 4.6, but works with 4.5
messages = []
add_user_message(messages, 'Generate a very short event bridge rule as json')
add_assistant_message(messages, '```json')
answer = chat(messages, stop_sequences=['```'])

print(answer)


{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["running"]
  }
}



In [37]:
messages = []
add_user_message(messages, 'Generate three different sample AWS CLI commands. Each should be very short')
answer = chat(messages)

print(answer)

Here are three short AWS CLI commands:

```bash
aws s3 ls

aws ec2 describe-instances

aws iam list-users
```


In [38]:
# Here Claude can still decide to do things on its own if we just provide ```bash prefill.
# It might create three separate blocks, it might also add comments explaining each command.
# To prevent this we can add more information to the prefill message to guide Claude to the response we want.
messages = []
add_user_message(messages, 'Generate three different sample AWS CLI commands. Each should be very short')
add_assistant_message(messages, 'Here are all three commands in a single block without any comments:\n```bash')
answer = chat(messages, stop_sequences=['```'])

print(answer)


aws s3 ls

aws ec2 describe-instances

aws iam list-users



In [42]:
# Prefilling is deprecated in 4.6+ models.
# The new way of formatting outputs is by using the output_format param
class Lead(BaseModel):
  model_config = {'extra': 'forbid'} # adds additionalProperties: false
  # define whatever fields you need
  name: str
  email: str
  plan_interest: str
  demo_requested: bool
  demo_time: str | None = None

messages = []
add_user_message(messages, 'Extract the key info from this email: John Smith (john@acme.com) is interested in our Enterprise Plan and wants to schedule a demo for next Tuesday at 2 pm.')

answer = client.messages.create(
  model='claude-sonnet-4-6',
  max_tokens=max_tokens,
  messages=messages,
  output_config={
    'format': {
      'type': 'json_schema',
      'schema': Lead.model_json_schema()
    }
  },
)

lead = Lead.model_validate_json(answer.content[0].text)

print(lead.model_dump_json())




{"name":"John Smith","email":"john@acme.com","plan_interest":"Enterprise Plan","demo_requested":true,"demo_time":"Next Tuesday at 2 PM"}
